# 🚀 LLooM Lab: Kaggle T4 Accelerator Runner (Verified October 2026)

This notebook executes the local model workloads for **LLooM Lab** using Kaggle's GPU acceleration (Tesla T4).

### ⚙️ Pre-Run Checklist (Right Sidebar -> Notebook Options):
1. **Accelerator**: Select **GPU T4 x2** (or **GPU T4**)
2. **Internet**: Toggle **Internet ON** (Required to download prebuilt wheels & embeddings)
3. **Inputs Attached**:
   - 📦 **Datasets**: `AML dataset S4.1` (or your uploaded dataset containing `aml-data-cache` and `aml-lloom-lab`)
   - 🧠 **Models**: `Qwen3 8B` (containing `qwen3-8b-q4_K_M.gguf`)

*This notebook dynamically discovers all files and models regardless of how Kaggle names dataset folders.*


In [ ]:
# 1. Hardware & Internet Verification
import os, sys, urllib.request, subprocess

print("=== Environment Check ===")
print("Python Version:", sys.version.split()[0])

# Check GPU
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ GPU Detected: {smi}")
except Exception as e:
    print("⚠️ GPU not detected! In the right sidebar, go to 'Notebook options' -> 'Accelerator' and select 'GPU T4'!")

# Check Internet
try:
    urllib.request.urlopen("https://www.google.com", timeout=5)
    print("✅ Internet Connection: Active")
except Exception:
    raise ConnectionError("❌ Internet is OFF! In the right sidebar under 'Notebook options', toggle 'Internet' to ON, then re-run.")

# List attached inputs
print("\n=== Mounted /kaggle/input ===")
for item in os.listdir("/kaggle/input"):
    print(f"  📁 /kaggle/input/{item}")


In [ ]:
# 2. Workspace Setup & Auto-Discovery
import glob, shutil
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/lloom-lab")

print("🔍 Auto-discovering project files in /kaggle/input...")

# 1. Locate GGUF Model
gguf_candidates = list(INPUT_ROOT.rglob("*.gguf"))
if not gguf_candidates:
    raise FileNotFoundError("❌ No .gguf model found in /kaggle/input! Please ensure your Qwen model is added to the notebook inputs.")
MODEL_GGUF_PATH = gguf_candidates[0]
print(f"✅ Model found: {MODEL_GGUF_PATH} ({MODEL_GGUF_PATH.stat().st_size / 1e9:.2f} GB)")

# 2. Locate Codebase (pyproject.toml)
code_candidates = list(INPUT_ROOT.rglob("pyproject.toml"))
if not code_candidates:
    raise FileNotFoundError("❌ Could not find pyproject.toml in /kaggle/input! Make sure AML dataset S4.1 / aml-lloom-lab is attached.")
SRC_ROOT = code_candidates[0].parent
print(f"✅ Codebase found at: {SRC_ROOT}")

# 3. Locate Corpus (docs.jsonl)
doc_candidates = list(INPUT_ROOT.rglob("docs.jsonl"))
if not doc_candidates:
    raise FileNotFoundError("❌ Could not find docs.jsonl in /kaggle/input! Make sure aml-data-cache is attached.")
DOCS_SRC = doc_candidates[0]
print(f"✅ Corpus found at: {DOCS_SRC}")

# 4. Locate Pre-existing Cache DB (optional)
cache_candidates = list(INPUT_ROOT.rglob("llm_cache.sqlite3"))

# 5. Populate /kaggle/working/lloom-lab
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

print(f"\nCopying codebase to {WORK_DIR}...")
shutil.copytree(SRC_ROOT, WORK_DIR, ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", ".ruff_cache", "*.tmp"))

# Ensure directory layout
DATA_STORE_DEST = WORK_DIR / "data_store" / "synthetic_v1"
DATA_STORE_DEST.mkdir(parents=True, exist_ok=True)
shutil.copy(DOCS_SRC, DATA_STORE_DEST / "docs.jsonl")

# Copy other files from synthetic_v1 if present (manifest, etc.)
for p in DOCS_SRC.parent.glob("*"):
    if p.is_file() and p.name != "docs.jsonl":
        shutil.copy(p, DATA_STORE_DEST / p.name)

RESULTS_DEST = WORK_DIR / "results"
RESULTS_DEST.mkdir(parents=True, exist_ok=True)
if cache_candidates:
    shutil.copy(cache_candidates[0], RESULTS_DEST / "llm_cache.sqlite3")
    print(f"✅ Imported existing LLM cache ({cache_candidates[0].stat().st_size / 1024:.1f} KB)")
else:
    print("ℹ️ Starting with a fresh LLM cache DB.")

# Ensure pyproject.toml compatibility (>=3.10)
pyproj_file = WORK_DIR / "pyproject.toml"
if pyproj_file.exists():
    t = pyproj_file.read_text(encoding="utf-8")
    t = t.replace('requires-python = ">=3.11"', 'requires-python = ">=3.10"')
    pyproj_file.write_text(t, encoding="utf-8")

print("\nInstalling dependencies in editable mode...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(WORK_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bertopic", "umap-learn", "fastembed"], check=True)
print("✅ Project workspace and dependencies successfully installed!")


In [ ]:
# 3. Install GPU-Accelerated llama-cpp-python & Server Extras
import subprocess, sys

print("📦 Installing llama-cpp-python with CUDA acceleration...")
# Install prebuilt wheel for CUDA 12.1 (standard Kaggle T4 driver)
install_cmd = [
    sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir",
    "llama-cpp-python",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"
]
subprocess.run(install_cmd, check=True)

# Install required server dependencies (including starlette-context & sse-starlette)
server_deps = [
    sys.executable, "-m", "pip", "install", "-q",
    "starlette-context", "sse-starlette", "pydantic-settings",
    "fastapi", "uvicorn", "httpx"
]
subprocess.run(server_deps, check=True)

# Verify imports
import llama_cpp
import starlette_context
import sse_starlette
print(f"✅ Successfully installed llama-cpp-python v{llama_cpp.__version__} with full server support!")


In [ ]:
# 4. Start 16GB VRAM-Accelerated llama-server
import os, time, requests, subprocess
from pathlib import Path

PORT = 8123
SERVER_URL = f"http://127.0.0.1:{PORT}/v1"
LOG_PATH = Path("/kaggle/working/llama_server.log")

# Check if already running
already_running = False
try:
    r = requests.get(f"{SERVER_URL}/models", timeout=2)
    if r.status_code == 200:
        already_running = True
        print(f"✅ Server is already running and ready on port {PORT}!")
except Exception:
    pass

if not already_running:
    print(f"🚀 Launching llama_cpp.server for model: {MODEL_GGUF_PATH.name}...")
    log_file = open(LOG_PATH, "w", buffering=1)
    
    server_cmd = [
        sys.executable, "-m", "llama_cpp.server",
        "--model", str(MODEL_GGUF_PATH),
        "--n_gpu_layers", "-1",       # Fully offload all layers into Tesla T4 VRAM
        "--n_ctx", "8192",             # 8K context length
        "--host", "127.0.0.1",
        "--port", str(PORT)
    ]
    
    proc = subprocess.Popen(server_cmd, stdout=log_file, stderr=subprocess.STDOUT)
    
    print("⏳ Waiting for model weights to load into GPU VRAM (usually ~15-25 seconds)...")
    is_ready = False
    for attempt in range(45):
        time.sleep(2)
        if proc.poll() is not None:
            print(f"❌ Server process exited with code {proc.returncode}!")
            log_file.close()
            print("--- Server Log Output ---")
            print(LOG_PATH.read_text(encoding="utf-8", errors="replace"))
            raise RuntimeError("llama_cpp.server crashed on startup. See log above.")
        try:
            resp = requests.get(f"{SERVER_URL}/models", timeout=2)
            if resp.status_code == 200:
                is_ready = True
                print(f"✅ llama-server is online and responsive! (ready in {(attempt + 1) * 2}s)")
                break
        except Exception:
            if (attempt + 1) % 5 == 0:
                print(f"   ... still loading ({ (attempt + 1) * 2 }s elapsed) ...")
                
    if not is_ready:
        log_file.close()
        print("--- Log Tail ---")
        print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-2000:])
        raise TimeoutError("Server failed to become ready within 90 seconds.")

# Smoke Test Inference
print("\n🧪 Running smoke test inference...")
test_resp = requests.post(
    f"{SERVER_URL}/chat/completions",
    json={
        "model": "qwen3",
        "messages": [{"role": "user", "content": "Return valid JSON: {\"status\": \"ready\"}"}],
        "temperature": 0.0,
        "max_tokens": 32
    },
    timeout=30
)
test_resp.raise_for_status()
print("Smoke test result:", test_resp.json()["choices"][0]["message"]["content"])
print("\n🎉 GPU Inference Engine is 100% OPERATIONAL!")


In [ ]:
# 5. Execute LLooM Local Experiment Cells
import os, subprocess, sys

os.chdir("/kaggle/working/lloom-lab")

# Run local cells for lloom_qwen across n in [10, 20, 40, 80, 160] with 2 parallel workers
print("🚀 Starting E1 local cells (lloom_qwen)...")
cmd = [
    sys.executable, "kaggle/run_local_cells.py",
    "--method", "lloom_qwen",
    "--ns", "10", "20", "40", "80", "160",
    "--trials", "5",
    "--jobs", "2"
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

print(f"\n✅ Execution finished with code {proc.returncode}!")


In [ ]:
# 6. (Optional) Run E4 Generalization Gap Benchmark
# Scores training vs held-out documents across induced concepts
import subprocess, sys

print("🚀 Running E4 (Uniform Convergence / Generalization Gap)...")
cmd = [
    sys.executable, "-m", "lloom_lab.experiments.e4",
    "--methods", "lloom_qwen",
    "--ns", "40", "80", "160",
    "--trials", "5",
    "--model", "qwen"
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

print(f"\n✅ E4 finished with code {proc.returncode}!")


In [ ]:
# 7. Package Results & Cache for 1-Click Download
import shutil, os
from pathlib import Path

out_archive = Path("/kaggle/working/kaggle_results")
source_dir = Path("/kaggle/working/lloom-lab/results")

print(f"📦 Compressing results from {source_dir}...")
archive_path = shutil.make_archive(str(out_archive), "zip", str(source_dir))

size_mb = os.path.getsize(archive_path) / (1024 * 1024)
print(f"\n✅ Successfully generated output archive: {archive_path} ({size_mb:.2f} MB)")
print("\n📥 TO DOWNLOAD:")
print("1. Look at the right sidebar under 'Output' -> '/kaggle/working'.")
print("2. Click the three dots next to 'kaggle_results.zip' and select 'Download'!")
print("3. On your local machine, run:")
print("   python lloom-lab/scripts/merge_cache.py path/to/kaggle_results/llm_cache.sqlite3")
